# Dairy_data_cleaning 

1.Parses the five FMMO December-2025 producer-milk table extracts in `raw/`, merges them into one
county panel (milk summed across orders), and reconciles every order×state total against the totals
printed in the original reports. Outputs land in `processed/`. 

2.Parses the FMMO order stats extract in `raw/` for each of the five orders and class I-IV use percentage shares. Processed file path is `processed/class_utilization_2025`.

3.Fetches the list of regulated supply and distibution plants list from the federal Datamart API which filters six states and decemner 2025. This file is then saved as CSV in the following paths: `raw/supply_plants_six_states_dec2025` and `raw/distribution_plants_six_states_dec2025` respectively

4.Uses the two plants list saved in step 3 with the city-county crosswalk, county center file, and census boundary file available in the raw folder to add infomation on the geographical location of the plants. The processed files are saved as CSV in the following paths: `processed/supply_plants_dec2025` and `processed/distribution_plants_six_dec2025` respectively



In [1]:
# Locate the repository path/root 
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / "raw").exists():
    ROOT = ROOT.parent
assert (ROOT / "raw").exists(), f"repository root not found from {Path.cwd()}"
print("repository root:", ROOT)

repository root: C:\Users\aakansha\Box\AAEA2026_Dataviz\AAEA2026_Dataviz_Dairy\aaea_dairy_viz_2


**Parse FMMO 30 (Upper Midwest)**

In [2]:
# Name of source and output file
SRC = ROOT / "raw/FMMO30_dec2025_extract.txt"
OUT = ROOT / "processed/fo30_dec2025_parsed.csv"

import re
from pathlib import Path
import pandas as pd


# Name of the states and their FIPS
STATE_FIPS = {8: "Colorado", 17: "Illinois", 18: "Indiana", 19: "Iowa", 21: "Kentucky",
              26: "Michigan", 27: "Minnesota", 29: "Missouri", 38: "North Dakota",
              39: "Ohio", 46: "South Dakota", 48: "Texas", 55: "Wisconsin"}

# December totals to match values at the end
PRINTED_ORDER_TOTAL = 1_199_765_312
PRINTED_ORDER_PRODUCERS = 3_036

# These three lines build templates that describes what each line in FO 30 tables looks like to sort between county, restricted, or grand total rows
ROW = re.compile(r"^(?P<pre>.*?)\s+(?P<sc>\d{1,2})\s+(?P<cc>\d{1,3})\s+"
                 r"(?P<prod>R|\d+)\s+(?P<milk>R|[\d,]+)\s*$")
REST = re.compile(r"^\s*Restricted\s+(?P<prod>\d+)\s+(?P<milk>[\d,]+)\s*$")
GRAND = re.compile(r"^\s*FO 30 Total\s+(?P<prod>[\d,]+)\s+(?P<milk>[\d,]+)\s*$")


def _num(x):
    return None if x == "R" else int(x.replace(",", ""))

# It reads the FO 30 file one line at a time, decides what each line is, and files it in the right bucket.
def parse():
    rows, restricted_lines, grand = [], [], None
    for line in SRC.read_text().splitlines():
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        if any(t in line for t in ("Page", "Federal Milk Order", "Table 15")):
            continue
        if "State" in line and "County" in line and "Code" in line:
            continue
        g = GRAND.match(line)
        if g:
            grand = (int(g["prod"].replace(",", "")), int(g["milk"].replace(",", "")))
            continue
        r = REST.match(line)
        if r:
            restricted_lines.append(_num(r["milk"]))
            continue
        m = ROW.match(line)
        if not m:
            continue
        sc, cc = int(m["sc"]), int(m["cc"])
        if sc not in STATE_FIPS:
            continue
        name = m["pre"].strip()
        for sn in ["North Dakota", "South Dakota", "New Mexico", *STATE_FIPS.values()]: #Name cleaning loop because of two-word names
            if name.startswith(sn + " "):
                name = name[len(sn):].strip()
                break
            if name == sn:
                name = ""
                break
        name = re.sub(r"\(continued\)", "", name).strip()
        rows.append(dict(order=30, order_name="Upper Midwest", year=2025, month="December", 
                         state_fips=sc, county_fips=cc, geo_fips=sc * 1000 + cc,   # Creates the county fips
                         state_name=STATE_FIPS[sc], county_name=name,
                         producers=_num(m["prod"]), milk_pounds=_num(m["milk"])))

    df = pd.DataFrame(rows)
    mapped = df.loc[df.milk_pounds.notna(), "milk_pounds"].sum()
    restricted = sum(restricted_lines)
    assert grand is not None, "FO 30 grand total row not found"
    assert grand[1] == PRINTED_ORDER_TOTAL, f"order total mismatch: {grand[1]}"
    assert mapped + restricted == PRINTED_ORDER_TOTAL, (
        f"reconciliation failed: {mapped:,} + {restricted:,} != {PRINTED_ORDER_TOTAL:,}")
    df.to_csv(OUT, index=False)
    print(f"[FO30] {len(df)} county rows | mapped {mapped:,} + restricted {restricted:,} " 
          f"= {PRINTED_ORDER_TOTAL:,}  -> {OUT.name}")
    return df # rows holds every county, restricted_lines holds the suppressed pounds, grand holds the printed total 

In [3]:
fo30 = parse()

[FO30] 234 county rows | mapped 1,143,871,119.0 + restricted 55,894,193 = 1,199,765,312  -> fo30_dec2025_parsed.csv


**Parse FMMO 33 (Mideast) — Michigan + western NY/PA**

In [4]:
RAW = ROOT / "raw"
OUT = ROOT / "processed"

from pathlib import Path
import pandas as pd


# printed state totals and their restricted (suppressed) components, from the report
PRINTED = {"MI": dict(fips=26, mapped=824_638_921, restricted=5_168_684, total=829_807_605),
           "NY": dict(fips=36, mapped=53_262_452, restricted=8_628_023, total=61_890_475),
           "PA": dict(fips=42, mapped=79_532_507, restricted=946_518, total=80_479_025)}


def _read(path):
    return [ln.split() for ln in path.read_text().splitlines()
            if ln.strip() and not ln.startswith("#")]

# All three input files are already well formatted. We use the codes below to generate clean CSV's from each extract

def parse_michigan():
    rows = []
    for p in _read(RAW / "FMMO33_MI_dec2025_extract.txt"):
        name, code, prod, milk = p[0], int(p[1]), int(p[2]), int(p[3].replace(",", "")) # unpack the four fields: county name, code, producers, pounds (commas stripped)
        rows.append(dict(order=33, order_name="Mideast", state_fips=26, county_fips=code,
                         geo_fips=26000 + code, state_name="Michigan", 
                         county_name=name.title(), producers=prod, milk_pounds=milk))
    df = pd.DataFrame(rows)
    _check("MI", df) #sums the table's pounds, checks for equality with the printed total
    df.to_csv(OUT / "fo33_mi_dec2025_parsed.csv", index=False)
    return df


def parse_ny_pa():
    rows = []
    for p in _read(RAW / "FMMO33_NY_PA_dec2025_extract.txt"):
        st, sfips, name, code, prod, milk = p[0], int(p[1]), p[2], int(p[3]), int(p[4]), int(p[5].replace(",", ""))
        rows.append(dict(order=33, order_name="Mideast", state_fips=sfips, county_fips=code,
                         geo_fips=sfips * 1000 + code, state_abbrev=st,
                         county_name=name.title(), producers=prod, milk_pounds=milk))
    df = pd.DataFrame(rows)
    for st in ("NY", "PA"):
        _check(st, df[df.state_abbrev == st])
    df.to_csv(OUT / "fo33_nypa_dec2025_parsed.csv", index=False)
    return df


def _check(st, df):
    mapped = int(df.milk_pounds.sum())
    exp = PRINTED[st]
    assert mapped == exp["mapped"], f"{st} mapped {mapped:,} != {exp['mapped']:,}"
    assert mapped + exp["restricted"] == exp["total"], f"{st} total mismatch"
    print(f"[FO33 {st}] {len(df)} counties | mapped {mapped:,} + restricted {exp['restricted']:,} "
          f"= {exp['total']:,}")

In [5]:
parse_michigan()
parse_ny_pa()

[FO33 MI] 47 counties | mapped 824,638,921 + restricted 5,168,684 = 829,807,605
[FO33 NY] 7 counties | mapped 53,262,452 + restricted 8,628,023 = 61,890,475
[FO33 PA] 31 counties | mapped 79,532,507 + restricted 946,518 = 80,479,025


,order,order_name,state_fips,county_fips,geo_fips,state_abbrev,county_name,producers,milk_pounds
0,33,Mideast,36,9,36009,NY,Cattaraugus,25,27613664
1,33,Mideast,36,13,36013,NY,Chautauqua,40,16362385
2,33,Mideast,36,29,36029,NY,Erie,5,1616348
3,33,Mideast,36,51,36051,NY,Livingston,4,945001
4,33,Mideast,36,69,36069,NY,Ontario,6,911405
5,33,Mideast,36,101,36101,NY,Steuben,3,1199577
6,33,Mideast,36,121,36121,NY,Wyoming,18,4614072
7,33,Mideast,42,5,42005,PA,Armstrong,9,2203191
8,33,Mideast,42,7,42007,PA,Beaver,9,540599
9,33,Mideast,42,9,42009,PA,Bedford,19,1027081


**Parse FMMO 32 (Central) and FMMO 5 (Appalachian)**

In [6]:
RAW = ROOT / "raw"
OUT = ROOT / "processed"

from pathlib import Path
import pandas as pd

# printed answer key, keyed by (order, state): mapped sum, restricted, state total
CHECK = {(32, "MN"): (6_879_970, 1_329_115, 8_209_085),
         (32, "WI"): (26_348_790, 3_529_555, 29_878_345),
         (5, "PA"): (10_538_018, 2_757_838, 13_295_856)}


def _parse(path, order, order_name):
    rows = []
    for ln in path.read_text().splitlines():
        if ln.startswith("#") or not ln.strip():
            continue
        st, sfips, name, code, prod, milk = ln.split()
        rows.append(dict(order=order, order_name=order_name, state_abbrev=st,
                         state_fips=int(sfips), county_fips=int(code),
                         geo_fips=int(sfips) * 1000 + int(code),
                         county_name=name.title(), producers=int(prod),
                         milk_pounds=int(milk)))
    return pd.DataFrame(rows)


def main():
    fo32 = _parse(RAW / "FMMO32_dec2025_extract.txt", 32, "Central")
    fo5 = _parse(RAW / "FMMO5_dec2025_extract.txt", 5, "Appalachian")
    for (order, st), (exp_map, restr, printed) in CHECK.items():
        df = {32: fo32, 5: fo5}[order]  # pick which table this check applies to
        got = int(df[df.state_abbrev == st].milk_pounds.sum())
        assert got == exp_map, f"FO{order} {st} mapped {got:,} != {exp_map:,}"
        assert got + restr == printed, f"FO{order} {st} total mismatch"   # mapped + suppressed must equal the printed state total
        print(f"[FO{order} {st}] {len(df[df.state_abbrev == st])} counties | "
              f"mapped {got:,} + restricted {restr:,} = {printed:,}")
    fo32.to_csv(OUT / "fo32_dec2025_parsed.csv", index=False)
    fo5.to_csv(OUT / "fo5_dec2025_parsed.csv", index=False)

In [7]:
main()

[FO32 MN] 6 counties | mapped 6,879,970 + restricted 1,329,115 = 8,209,085
[FO32 WI] 9 counties | mapped 26,348,790 + restricted 3,529,555 = 29,878,345
[FO5 PA] 12 counties | mapped 10,538,018 + restricted 2,757,838 = 13,295,856


**Parse FMMO 1 (Northeast) — annual statistical bulletin county table**

In [8]:
import re as _re
import json as _json

FO1_FILE = ROOT / "raw/FMMO1_northeast_dec2025.csv"
PROC = ROOT / "processed"
# printed December state totals from the bulletin, in pounds
_PRINTED_FO1 = {"New York": 1_211_317_000, "Vermont": 205_153_000, "Pennsylvania": 647_659_000}
_SFIPS = {"New York": "36", "Vermont": "50", "Pennsylvania": "42"}

# county name: FIPS lookup built from the county boundary file
_gj = _json.load(open(ROOT / "raw/us_counties.json"))
_name2fips = {}
for _f in _gj["features"]:
    _p = _f.get("properties", {})
    _fp = str(_p.get("GEOID") or _f.get("id")).zfill(5)  # county ID may be stored as GEOID or id; zfill restores leading zeros (9003 -> 09003)
    _name2fips[(_fp[:2], _re.sub(r"[^a-z]", "", str(_p.get("NAME")).lower()))] = _fp  # key = (state fips, name with punctuation stripped)

_STATES = {"Connecticut","Delaware","Maine","Maryland","Massachusetts","New Hampshire","New Jersey",
           "New York","Pennsylvania","Rhode Island","Vermont","Virginia"}  # bare state names act as section headings in the bulletin
_SKIP = ("Producer Receipts","October","December","State and county","No. of","producers",
         "of milk","1,000","(Continued","NORTHEAST","Total/Average","Grand Total","Other States","*") #line-starts that are headers/footers, not data

# a county line: name then oct_prod oct_vol nov_prod nov_vol dec_prod dec_vol annual_vol (1,000 lb)
_ROW = _re.compile(r"^(.+?)\s+(\d[\d,]*)\s+([\d,]+)\s+(\d[\d,]*)\s+([\d,]+)\s+(\d[\d,]*)\s+([\d,]+)\s+([\d,]+)$")

def parse_fo1():
    rows, state = [], None
    for raw in open(FO1_FILE, encoding="utf-8-sig"):   # -sig strips Excel's invisible leading character
        ln = raw.strip().strip(",").strip('"').replace("\r", "") ## undo the quotes/commas the CSV export wrapped around each line
        if not ln:
            continue
        if ln in _STATES:
            state = ln
            continue
        if ln.startswith(_SKIP):
            continue
        m = _ROW.match(ln)
        if not (m and state in _PRINTED_FO1):  # keep only county rows under NY/VT/PA
            continue
        name = m.group(1).strip()
        p_dec = int(m.group(6).replace(",", "")) # take only the December pair (groups 6-7); Oct/Nov/Annual are ignored
        lbs = int(m.group(7).replace(",", "")) * 1000  # bulletin reports volumes in 1,000-lb units
        # "All Other" = the state's suppressed counties, kept as a restricted row
        if name.startswith("All Other"):
            rows.append(dict(state_name=state, county_name=None, geo_fips=None,
                             producers=p_dec, milk_pounds=lbs, restricted=1))
        else:
            fp = _name2fips.get((_SFIPS[state], _re.sub(r"[^a-z]", "", name.lower())))
            assert fp, f"county not matched to FIPS: {state} / {name}"  # every county must match the map geometry, or stop
            rows.append(dict(state_name=state, county_name=name, geo_fips=int(fp),
                             producers=p_dec, milk_pounds=lbs, restricted=0))
    fo1 = pd.DataFrame(rows)
    for st, want in _PRINTED_FO1.items():
        got = int(fo1.loc[fo1.state_name == st, "milk_pounds"].sum())
        assert got == want, f"FO1 {st}: parsed {got:,} != printed {want:,}"
    fo1.to_csv(PROC / "fo1_dec2025_parsed.csv", index=False)
    print(f"[FO1] {int((fo1.restricted==0).sum())} county rows + "
          f"{int((fo1.restricted==1).sum())} restricted rows | "
          "all three state totals match the printed bulletin")
    return fo1

fo1_parsed = parse_fo1()

[FO1] 106 county rows + 2 restricted rows | all three state totals match the printed bulletin


**Compile all the six parsed order files**

In [15]:
RAW = ROOT / "raw"
PROC = ROOT / "processed"

from pathlib import Path
import numpy as np
import pandas as pd


SAB = {8: "CO", 9: "CT", 17: "IL", 18: "IN", 19: "IA", 21: "KY", 23: "ME", 24: "MD",
       25: "MA", 26: "MI", 27: "MN", 29: "MO", 33: "NH", 36: "NY", 38: "ND", 39: "OH",
       42: "PA", 46: "SD", 48: "TX", 50: "VT", 55: "WI"}
UM_STATES = {"MN", "WI", "MI"}
NE_STATES = {"NY", "VT", "PA"}

# 13 printed order-state totals; FO 1 absent because its parser already checked its own
PRINTED = {(30, "MN"): 252_480_542, (30, "WI"): 834_906_434, (30, "MI"): 7_382_162,
           (33, "MI"): 829_807_605, (33, "NY"): 61_890_475, (33, "PA"): 80_479_025,
           (33, "WI"): 3_267_185, (33, "MN"): 190_420,
           (32, "MN"): 8_209_085, (32, "WI"): 29_878_345, (32, "MI"): 2_427_818,
           (5, "PA"): 13_295_856, (5, "MI"): 33_195_682}
# per-state restricted (suppressed) aggregates that carry no county identity
RESTRICTED = {(30, 27): 27_352_442, (30, 55): 6_367_359,
              (33, 26): 5_168_684, (33, 36): 8_628_023, (33, 42): 946_518,
              (33, 55): 3_267_185, (33, 27): 190_420,
              (32, 27): 1_329_115, (32, 55): 3_529_555, (32, 26): 2_427_818,
              (5, 42): 2_757_838, (5, 26): 33_195_682}
ORDER_NAMES = {1: "Northeast", 5: "Appalachian", 30: "Upper Midwest",
               32: "Central", 33: "Mideast"}


def _pad(x):                                       # any FIPS -> 5-char text; None instead of crashing on blanks
    try:
        return str(int(x)).zfill(5)
    except (TypeError, ValueError):
        return None


def _name_lookup():                                # county names taken from the parsed files themselves
    ref = {}
    fo1 = pd.read_csv(PROC / "fo1_dec2025_parsed.csv")
    for _, r in fo1.dropna(subset=["geo_fips", "county_name"]).iterrows():
        ref[_pad(r["geo_fips"])] = r["county_name"]
    for f in ["fo30_dec2025_parsed.csv", "fo33_mi_dec2025_parsed.csv",
              "fo33_nypa_dec2025_parsed.csv", "fo32_dec2025_parsed.csv",
              "fo5_dec2025_parsed.csv"]:
        d = pd.read_csv(PROC / f)
        if "county_name" in d:
            for _, r in d.dropna(subset=["geo_fips", "county_name"]).iterrows():
                ref.setdefault(_pad(r["geo_fips"]), r["county_name"])
    return ref


def build_long():
    ref = _name_lookup()
    recs = []

    def add(order, oname, sfips, cfips, prod, milk, rtype):     # one standardized record; every source block feeds this
        gf = None if cfips is None else sfips * 1000 + cfips
        f5 = _pad(gf) if gf is not None else None
        recs.append(dict(order=order, order_name=oname, year=2025, month="December",
                         geo_fips=gf, state_fips=sfips, county_fips=cfips,
                         state_abbrev=SAB.get(sfips), county_name=(ref.get(f5) if f5 else None),
                         producers=prod, milk_pounds=milk, row_type=rtype,
                         mappable=int(rtype == "county" and pd.notna(milk)))) # 1 = row has both a county and a milk value, so it can appear on the map

    # FO 1 — NY/VT/PA from the parsed annual-bulletin table
    fo1 = pd.read_csv(PROC / "fo1_dec2025_parsed.csv")
    _fo1_sfips = {"New York": 36, "Vermont": 50, "Pennsylvania": 42}
    for _, r in fo1.iterrows():
        county = r["restricted"] == 0
        add(1, "Northeast", _fo1_sfips[r["state_name"]],
            int(r["geo_fips"]) % 1000 if county else None,     # % 1000 peels the county part off a 5-digit FIPS
            None if pd.isna(r["producers"]) else int(r["producers"]),
            None if pd.isna(r["milk_pounds"]) else int(r["milk_pounds"]),
            "county" if county else "restricted_aggregate")

    # FO 30 — MN/WI/MI county rows
    fo30 = pd.read_csv(PROC / "fo30_dec2025_parsed.csv")
    fo30 = fo30[fo30.state_abbrev.isin(UM_STATES)] if "state_abbrev" in fo30 else\
        fo30[fo30.state_name.isin(["Minnesota", "Wisconsin", "Michigan"])]
    for _, r in fo30.iterrows():
        add(30, "Upper Midwest", int(r["state_fips"]), int(r["county_fips"]),
            None if pd.isna(r["producers"]) else int(r["producers"]),
            None if pd.isna(r["milk_pounds"]) else int(r["milk_pounds"]), "county")

    # FO 33 — MI + western NY/PA county rows
    mi = pd.read_csv(PROC / "fo33_mi_dec2025_parsed.csv")
    for _, r in mi.iterrows():
        add(33, "Mideast", 26, int(r["geo_fips"]) - 26000, int(r["producers"]),
            int(r["milk_pounds"]), "county")
    nypa = pd.read_csv(PROC / "fo33_nypa_dec2025_parsed.csv")
    for _, r in nypa.iterrows():
        add(33, "Mideast", int(r["state_fips"]), int(r["county_fips"]),
            int(r["producers"]), int(r["milk_pounds"]), "county")

    # FO 32 (Central) — MN/WI mapped counties
    fo32 = pd.read_csv(PROC / "fo32_dec2025_parsed.csv")
    for _, r in fo32.iterrows():
        add(32, "Central", int(r["state_fips"]), int(r["county_fips"]),
            int(r["producers"]), int(r["milk_pounds"]), "county")

    # FO 5 (Appalachian) — PA mapped counties (MI/NY are fully restricted, added below)
    fo5 = pd.read_csv(PROC / "fo5_dec2025_parsed.csv")
    for _, r in fo5.iterrows():
        add(5, "Appalachian", int(r["state_fips"]), int(r["county_fips"]),
            int(r["producers"]), int(r["milk_pounds"]), "county")

    # restricted (suppressed) state aggregates, all orders
    for (order, sfips), pounds in RESTRICTED.items():
        add(order, ORDER_NAMES[order], sfips, None, None, pounds, "restricted_aggregate")

    return pd.DataFrame(recs)


def build_totals(long):
    m = long[long.mappable == 1].copy()
    tot = (m.groupby("geo_fips")
             .agg(state_abbrev=("state_abbrev", "first"),
                  county_name=("county_name", "first"),
                  total_producers=("producers", "sum"),
                  total_milk_pounds=("milk_pounds", "sum"),
                  n_orders=("order", "nunique"),
                  orders=("order", lambda s: "+".join(f"FO{o}" for o in sorted(s.unique()))))  # label like "FO30+FO32" for multi-order counties
             .reset_index())
    tot["region"] = np.select(
        [tot.state_abbrev.isin(UM_STATES), tot.state_abbrev.isin(NE_STATES)],
        ["Upper Midwest", "Northeast"], default="Other")                       # state -> region tag for the two panels
    return tot


def build_reconciliation(long):
    rows = []
    for (order, sa), printed in PRINTED.items():
        sub = long[(long.order == order) & (long.state_abbrev == sa)]
        mapped = int(sub.loc[sub.mappable == 1, "milk_pounds"].sum())
        restr = int(sub.loc[sub.row_type == "restricted_aggregate", "milk_pounds"].sum())
        rows.append(dict(order=order, state=sa, mapped=mapped, restricted=restr,
                         compiled_total=mapped + restr, printed_total=printed,
                         reconciles=(mapped + restr == printed)))
    rec = pd.DataFrame(rows)
    assert rec.reconciles.all(), "reconciliation failed:\n" + rec.to_string()
    return rec


def compile_all():
    long = build_long()
    tot = build_totals(long)
    rec = build_reconciliation(long)
    long.to_csv(PROC / "compiled_long_dec2025.csv", index=False)
    tot.to_csv(PROC / "county_total_dec2025.csv", index=False)
    rec.to_csv(PROC / "reconciliation_dec2025.csv", index=False)
    for reg, g in tot.groupby("region"):
        print(f"[{reg}] {g.geo_fips.nunique()} counties | {g.total_milk_pounds.sum()/1e6:.1f}M lbs")
    print(f"[reconciliation] all {len(rec)} order x state rows reconcile to printed totals")

In [16]:
compile_all()

[Northeast] 121 counties | 2204.5M lbs
[Upper Midwest] 152 counties | 1918.9M lbs
[reconciliation] all 13 order x state rows reconcile to printed totals


**Parse the 2025 class-utilization shares**

In [17]:
UTIL_FILE = ROOT / "raw/2025_fmmo_statistics_extract.txt"
PROC = ROOT / "processed"

# FO 30's December shares as printed in its own report (Table 9) — used as an outside check
_FO30_DEC = (14.73, 13.17, 58.49, 13.61)

def parse_utilization():
    rows = []
    for ln in UTIL_FILE.read_text().splitlines():
        if ln.startswith("#") or not ln.strip():
            continue
        p = ln.split()                                      ## 11 fields: order, name, receipts, 4 Dec shares, 4 annual shares
        rows.append(dict(order=int(p[0]), order_name=p[1], dec_receipts=int(p[2]),
                         dec_I=float(p[3]), dec_II=float(p[4]), dec_III=float(p[5]), dec_IV=float(p[6]),
                         ann_I=float(p[7]), ann_II=float(p[8]), ann_III=float(p[9]), ann_IV=float(p[10])))
    util = pd.DataFrame(rows)
    assert len(util) == 5 and set(util.order) == {1, 5, 30, 32, 33}, "expected the five orders" # all five orders must be present, no more, no less
    for _, r in util.iterrows():
        for pre in ("dec", "ann"):
            tot = sum(r[f"{pre}_{k}"] for k in ("I", "II", "III", "IV"))
            assert abs(tot - 100) < 0.15, f"FO{r.order} {pre} shares sum to {tot}" # each set of four shares must add to 100 (small rounding allowed)
    fo30 = util.loc[util.order == 30].iloc[0]
    assert tuple(fo30[["dec_I", "dec_II", "dec_III", "dec_IV"]]) == _FO30_DEC, \
        "FO30 December shares do not match CompStats Table 9"
    util.to_csv(PROC / "class_utilization_2025.csv", index=False)
    print("[utilization] 5 orders | Dec + annual Class I-IV shares parsed; "
          "all sums = 100, FO30 Dec matches CompStats Table 9")
    return util

util = parse_utilization()

[utilization] 5 orders | Dec + annual Class I-IV shares parsed; all sums = 100, FO30 Dec matches CompStats Table 9


**Download the AMS plant lists**

In [18]:
import requests

_AMS_3359 = {                                                                  # each plant type: (filename to cache under, USDA API address)
    "Supply": ("supply_plants_six_states_dec2025.csv",
               "https://mpr.datamart.ams.usda.gov/services/v1.1/reports/3359/Supply%20Plants%20by%20Month"),
    "Distributing": ("distributing_plants_six_states_dec2025.csv",
                     "https://mpr.datamart.ams.usda.gov/services/v1.1/reports/3359/Distributing%20Plants%20by%20Month"),
}
_SIX = {"MN", "WI", "MI", "NY", "PA", "VT"}

def fetch_ams_plants():
    for kind, (fname, url) in _AMS_3359.items():
        target = ROOT / "raw" / fname
        if target.exists():                                  # already downloaded once — reuse the saved copy
            print(f"[AMS 3359] using cached raw/{fname}")    
            continue
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()                                # stop with an error if the server didn't answer properly
        payload = resp.json()
        records = payload.get("results", payload) if isinstance(payload, dict) else payload  # the answer may come wrapped in {"results": ...} or bare
        d = pd.DataFrame(records)
        need = {"Plant_Name", "City", "State_code", "report_year", "dec"}
        assert need <= set(d.columns), f"unexpected API columns: {sorted(d.columns)}"  # if USDA changes their column names, stop and show what arrived
        # the API is wide: one row per plant-year, month columns hold that month's order code;
        # a blank December cell means the plant was not a regulated pool plant that month
        d = d[(pd.to_numeric(d.report_year, errors="coerce") == 2025)
              & (d.State_code.isin(_SIX))].copy()
        d["december_fmmo_order"] = pd.to_numeric(d["dec"], errors="coerce")
        d = d[d.december_fmmo_order.notna()].copy()
        d["december_fmmo_order"] = d["december_fmmo_order"].astype(int)
        assert set(d.december_fmmo_order) <= {1, 5, 30, 32, 33}, \
            f"unexpected December order codes: {sorted(d.december_fmmo_order.unique())}"
        tidy = pd.DataFrame({
            "report_year": 2025, "reference_month": "December", "plant_type": kind,
            "plant_name": d.Plant_Name.values, "city": d.City.values, "state": d.State_code.values,
            "december_fmmo_order": d.december_fmmo_order.values,
            "class_i_differential": d.get("classI_Diff", pd.Series(index=d.index)).values,  # optional columns: keep if present, blank if not
            "report_title": d.get("report_title", pd.Series(index=d.index)).values,
            "source_slug": "AMS_3359",
            "published_date": d.get("published_date", pd.Series(index=d.index)).values,
        })
        tidy.to_csv(target, index=False)                                       # cache in raw/ so later runs work without internet
        print(f"[AMS 3359] downloaded {kind} -> raw/{fname} ({len(tidy)} rows)")

fetch_ams_plants()

[AMS 3359] using cached raw/supply_plants_six_states_dec2025.csv
[AMS 3359] using cached raw/distributing_plants_six_states_dec2025.csv


In [21]:
import re as _re
import json as _json

SUP_RAW = ROOT / "raw/supply_plants_six_states_dec2025.csv"
DIS_RAW = ROOT / "raw/distributing_plants_six_states_dec2025.csv"
PROC = ROOT / "processed"

_ORDER_NAMES = {1: "FO1 Northeast", 5: "FO5 Appalachian", 30: "FO30 Upper Midwest",
                32: "FO32 Central", 33: "FO33 Mideast"}
_UM, _NE = {"MN", "WI", "MI"}, {"NY", "PA", "VT"}
_ST_FIPS = {"MN": "27", "WI": "55", "MI": "26", "NY": "36", "PA": "42", "VT": "50"}

# For three cities the crosswalk doesn't know; counties taken from the plants' published addresses
_MANUAL_COUNTY = {("Hollandtown", "WI"): "BROWN", ("Wiota", "WI"): "LAFAYETTE",
                  ("Woodbury", "MN"): "WASHINGTON"}


def _fold(s):                                   # make names comparable: lowercase, drop punctuation, "saint" -> "st"
    return _re.sub(r"\bsaint\b", "st", _re.sub(r"[^a-z0-9 ]", "", str(s).lower())).replace(" ", "")


def build_plants():
    cw = pd.read_csv(ROOT / "raw/city_county_crosswalk.csv", sep="|")
    cw["k"] = cw["City"].map(_fold) + "|" + cw["State short"]
    city2county = cw.drop_duplicates("k").set_index("k")["County"].to_dict()    # first entry wins when a city appears more than once

    gj = _json.load(open(ROOT / "raw/us_counties.json"))
    name2fips = {}
    for f in gj["features"]:
        p = f.get("properties", {})
        fp = str(p.get("GEOID") or f.get("id")).zfill(5)
        name2fips[(fp[:2], _fold(p.get("NAME")))] = fp          # key = (state, folded county name), so "ST. CROIX" finds "Saint Croix"

    centers = pd.read_csv(ROOT / "raw/county_centers.csv", dtype={"fips": str})
    fips2ll = centers.set_index("fips")[["pclat10", "pclon10"]].to_dict("index")  # population-weighted county centers (2010)

    out = {}
    for kind, path in [("Supply", SUP_RAW), ("Distributing", DIS_RAW)]:
        d = pd.read_csv(path)
        assert (d.report_year == 2025).all() and (d.reference_month == "December").all() \
            and (d.plant_type == kind).all() and set(d.state) <= set(_ST_FIPS), \
            f"{path.name}: unexpected content"
        rows = [dict(plant_name=r.plant_name, city=r.city, state=r.state,
                     dec_order=int(r.december_fmmo_order),
                     validation="AMS 3359 December column")
                for _, r in d.iterrows()]
        for r in rows:
            county = _MANUAL_COUNTY.get((r["city"], r["state"])) \
                or city2county.get(_fold(r["city"]) + "|" + r["state"])          # manual fix first, then the crosswalk
            assert county, f"no county for {r['plant_name']} ({r['city']}, {r['state']})" # a plant we can't place stops the run — no silent drops
            fp = name2fips.get((_ST_FIPS[r["state"]], _fold(county)))
            assert fp, f"no FIPS for {county}, {r['state']}"
            ll = fips2ll[fp]
            r.update(county=county, county_fips=int(fp), lat=ll["pclat10"], lon=ll["pclon10"],
                     dec_order_name=_ORDER_NAMES[r["dec_order"]],
                     region="Upper Midwest" if r["state"] in _UM else "Northeast",
                     pooled_dec2025=True, on_dec_map=True)
        out[kind] = pd.DataFrame(rows)

    sup, dis = out["Supply"], out["Distributing"]
    counts = {(k, reg): int((df.region == reg).sum())
              for k, df in out.items() for reg in ("Upper Midwest", "Northeast")}
    assert counts == {("Supply", "Upper Midwest"): 40, ("Supply", "Northeast"): 8,
                      ("Distributing", "Upper Midwest"): 16, ("Distributing", "Northeast"): 34}, counts
    sup.to_csv(PROC / "supply_plants_dec2025.csv", index=False)
    dis.to_csv(PROC / "distributing_plants_dec2025.csv", index=False)
    print("[plants] supply 40 UM + 8 NE, distributing 16 UM + 34 NE; all geocoded")
    return sup, dis

supply_plants, distributing_plants = build_plants()

[plants] supply 40 UM + 8 NE, distributing 16 UM + 34 NE; all geocoded
